# 01 · Data prep: from raw recordings to one clean dataset

**Run this first.** By the end you will have built, yourself, the clean file every other notebook loads
(`data/processed_mini.npz`), and you will know what is in it and why.

**How this notebook works**
- Cells marked **`# TODO`** are yours to fill in. Each one has a docstring that says the input and output shapes.
- **Stuck?** Under every TODO there are three **💡 hints** that open one at a time: the plan in words, then the
  functions you need with a tiny example, then a skeleton with a few blanks (`____`) left. Open only as many as
  you need. A good habit: add a scratch cell (`+ Code`) and try each line on a small made-up array first.
- Right after each TODO there is a **✅ check cell**. Run it: it compares your answer with the team's reference
  code and tells you what is wrong. Then a **📊 plot cell** shows what your code did to the data.
- The finished reference lives in [`prep.py`](prep.py). Notebooks 02–06 call it, so your blanks never break
  anyone else's notebook. Try each task before you look at it.
- Questions marked **✍️** are for you to answer in your own words in the markdown cell below them.

**The pipeline you will build**
1. Open a raw recording and see what is inside (16 EMG channels at 2,000 samples/s, 20 joint angles)
2. Turn raw EMG into a smooth **envelope** (how active each muscle area is) at 50 samples/s
3. Mark frames where pose tracking failed
4. Split each recording in time: first 80% train, 1 s gap, last 20% test
5. Run it on all 30 recordings, check against the reference, and explore the clean dataset

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ✅ preprocessing (weeks 1–5). The proposal's NMF runs on *rectified, amplitude-normalised envelopes*: this notebook makes them.

**Goal:** give the whole team one identical, clean dataset, and make sure each of us understands it.

**Why:** if each person filters or splits the data their own way, our numbers can't be compared. And if we don't
understand the data, we can't tell a real finding from a preprocessing artefact.

**Milestones**
1. Every ✅ check cell passes
2. You can explain the envelope (filter → rectify → smooth → downsample → scale) to the team in 1 minute
3. You can say what one row of `processed_mini.npz` is, and what each array in it means
4. The ✍️ questions are answered

**Done when:** the final check says your dataset matches the reference.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

PREP_DIR = ROOT / "notebooks" / "experiments" / "01_data_prep"
assert (PREP_DIR / "prep.py").exists(), (
    f"{PREP_DIR}/prep.py is missing. On Colab this means it was never pushed to GitHub; "
    "otherwise Python would silently import some other module called prep.")
sys.path.insert(0, str(PREP_DIR))
sys.modules.pop("prep", None)            # forget any other module named "prep" imported earlier
import prep as ref          # the reference answers, used only by the ✅ check cells
assert Path(ref.__file__).resolve().parent == PREP_DIR, f"Imported the wrong prep module: {ref.__file__}"

PROCESSED = ROOT / "data" / "processed_mini.npz"
DATA_DIR = ref.find_data_dir(ROOT)      # downloads the mini dataset (~617 MB) the first time
print(f"{'Colab' if IN_COLAB else 'Local'} | repo root: {ROOT} | data: {DATA_DIR.relative_to(ROOT)}")

In [ ]:
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

FS_OUT = 50          # samples per second after downsampling
TRAIN_FRACTION = 0.8
GAP = FS_OUT         # 1 s gap between train and test, so smoothing can't leak across

# The 20 joint angles, in column order (from emg2pose/constants.py).
# FE = flexion/extension (bending), AA = abduction/adduction (spreading).
# CMC/MCP = knuckle near the palm, PIP = middle knuckle, DIP/IP = knuckle near the tip.
JOINT_NAMES = [
    "THUMB_CMC_FE", "THUMB_CMC_AA", "THUMB_MCP_FE", "THUMB_IP_FE",
    "INDEX_MCP_AA", "INDEX_MCP_FE", "INDEX_PIP_FE", "INDEX_DIP_FE",
    "MIDDLE_MCP_AA", "MIDDLE_MCP_FE", "MIDDLE_PIP_FE", "MIDDLE_DIP_FE",
    "RING_MCP_AA", "RING_MCP_FE", "RING_PIP_FE", "RING_DIP_FE",
    "PINKY_MCP_AA", "PINKY_MCP_FE", "PINKY_PIP_FE", "PINKY_DIP_FE",
]

files = sorted(glob.glob(str(DATA_DIR / "*.hdf5")))
print(f"{len(files)} recordings found")

---
## Part 1 · What is inside one recording?

Each `.hdf5` file is **one hand doing one task ("stage")**, usually for about a minute (in the mini dataset they
run from 46 s to 4 min). HDF5 is a file format that stores arrays in folders, like a tiny file system.
Inside each file:

```
emg2pose/                     ← a group (folder); its only item is "timeseries"
│   .attrs                    ← labels stuck ON the group (not an item inside it):
│                               stage, side ("left"/"right"), user, sample_rate, split, …
└── timeseries                ← ONE array of T rows; each row has three named fields:
        time          (T,)        seconds (Unix time)
        joint_angles  (T, 20)     hand pose, 20 angles in RADIANS
        emg           (T, 16)     raw EMG, 16 electrodes around the wrist
```

`timeseries` is a *structured array*: you pick a field by name, e.g. `ts["emg"]`. Labels are read with
`g.attrs["stage"]`, not `g["stage"]`. Run the cell below to look.

In [ ]:
# 📊 Explore: just run this and read the output
with h5py.File(files[0], "r") as f:
    print("Top-level keys:", list(f.keys()))
    g = f["emg2pose"]
    print("Inside emg2pose:", list(g.keys()))
    print("\nLabels (attrs):")
    for k, v in g.attrs.items():
        print(f"  {k:16s} {v}")
    ts = g["timeseries"]
    print("\ntimeseries dtype:", ts.dtype)
    print("timeseries shape:", ts.shape, "rows")
    print("\nFirst 3 time stamps:", ts["time"][:3])
    print("Gap between samples (s):", np.diff(ts["time"][:3]))

### 🧰 The NumPy you need for every task
All the data here are **2-D arrays: rows = time, columns = channels** (or joints). Five ideas cover every task:

| Idea | Code | Means |
|---|---|---|
| Shape | `x.shape` → `(T, 16)` | T rows (time steps), 16 columns (channels) |
| Slice rows | `x[10:20]`, `x[:100]`, `x[::40]` | rows 10–19 · first 100 rows · every 40th row |
| Pick a column | `x[:, 3]` | all rows, column 3 (one channel over time) |
| `axis=` | `x.mean(axis=0)` → (16,) · `x.mean(axis=1)` → (T,) | axis 0 = *down the rows* (over time), axis 1 = *across the columns* (per row) |
| Masks | `m = x[:, 0] > 1` then `x[m]` | a True/False per row; keeps the True rows. Combine with `&` (and), `\|` (or), `~` (not) |

Run the cell below and predict each printed shape *before* you read it.

In [ ]:
# 🧰 Play with a tiny fake recording: 6 time steps × 3 channels
x = np.arange(18, dtype=float).reshape(6, 3)
print(x, "\n")
print("shape:", x.shape)
print("first 2 rows:\n", x[:2])
print("every 2nd row:\n", x[::2])
print("column 1 over time:", x[:, 1])
print("mean of each column (axis=0):", x.mean(axis=0))
print("mean of each row (axis=1):", x.mean(axis=1))
m = x[:, 0] > 6
print("mask:", m, "→ kept rows:\n", x[m])
print("~mask:", ~m)

### Task 1 · Load a recording
Write `load_recording`. Two details matter:
- The sample rate is not stored per row, so compute it from the time stamps: `1 / (typical gap between samples)`.
  Use the **median** gap of the first 1,000 samples, so one odd gap can't throw it off.
- Convert joint angles from radians to **degrees** (`np.rad2deg`), since degrees are easier to read.

In [ ]:
def load_recording(path: str) -> tuple[np.ndarray, np.ndarray, float, str, str]:
    """Load one raw recording.

    Returns
    -------
    emg_raw  : (T, 16) float32, raw EMG
    pose_deg : (T, 20) float32, joint angles in DEGREES
    fs       : float, samples per second (should come out as 2000.0)
    stage    : str, the task name, from attrs["stage"]
    side     : str, "left" or "right", from attrs["side"]

    Hints: open with `h5py.File(path, "r")`, then `g = f["emg2pose"]`, `ts = g["timeseries"]`.
    `ts["emg"][:]` reads the whole field into memory. Wrap attrs in str(...).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Open the file (`with h5py.File(path, "r") as f:`). Everything that reads from it goes *inside* the `with` block,
   because the file closes when the block ends.
2. Go down the folders: `f["emg2pose"]`, then its `"timeseries"`.
3. Read the `emg` field and the `joint_angles` field into NumPy arrays; convert the angles to degrees.
4. Sample rate: the gaps between time stamps are about 0.0005 s, so fs = 1 / 0.0005 = 2000.
5. Read `stage` and `side` from `.attrs` and return the five things in the order the docstring lists.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `ts["emg"][:]` the `[:]` reads the whole field from disk into memory as a NumPy array.
- `.astype(np.float32)` stores numbers in 32 bits (half the memory of the default 64).
- `np.rad2deg(a)` radians → degrees. `np.diff(a)` gaps between neighbours. `np.median(a)` the middle value.
- `g.attrs["stage"]` reads a label; wrap it in `str(...)`.

```python
t = np.array([0.0, 0.0005, 0.0010, 0.0015])
np.diff(t)                # array([0.0005, 0.0005, 0.0005])
1 / np.median(np.diff(t)) # 2000.0
np.rad2deg(np.pi / 2)     # 90.0
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def load_recording(path: str) -> tuple[np.ndarray, np.ndarray, float, str, str]:
    with h5py.File(path, "r") as f:
        g = f["emg2pose"]
        ts = g["timeseries"]
        emg_raw = ts["emg"][:].astype(np.float32)
        pose_deg = np.rad2deg(ts[____][:].astype(np.float32))
        gaps = np.diff(ts["time"][:1000])
        fs = float(1.0 / ____)
        return emg_raw, pose_deg, fs, str(g.attrs["stage"]), str(____)
```

</details>

In [ ]:
# ✅ Check Task 1
emg_raw, pose_deg, fs, stage, side = load_recording(files[0])
r_emg, r_pose, r_fs, r_stage, r_side = ref.load_recording(files[0])
assert emg_raw.shape == r_emg.shape, f"emg_raw shape {emg_raw.shape}, expected {r_emg.shape}"
assert pose_deg.shape == r_pose.shape, f"pose_deg shape {pose_deg.shape}, expected {r_pose.shape}"
assert np.allclose(pose_deg, r_pose), "pose values differ: did you convert radians to degrees?"
assert abs(fs - r_fs) < 1e-3, f"fs = {fs}, expected {r_fs}"
assert (stage, side) == (r_stage, r_side)
print(f"✅ {stage}, {side} hand: {len(emg_raw):,} samples = {len(emg_raw)/fs:.0f} s at {fs:.0f} Hz")

### 📊 Look at the raw data
Top: 2 seconds of all 16 EMG channels, stacked. Bottom: all 20 joint angles over the whole recording.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# EMG: 2 s, one line per channel, shifted up so they don't overlap
n = int(2 * fs)
t = np.arange(n) / fs
spacing = 4 * emg_raw[:n].std()
for ch in range(16):
    ax[0].plot(t, emg_raw[:n, ch] + ch * spacing, lw=0.4, color="k")
ax[0].set_yticks(np.arange(16) * spacing, [f"ch{ch}" for ch in range(16)])
ax[0].set_xlabel("time (s)"); ax[0].set_title(f"Raw EMG, first 2 s ({fs:.0f} samples/s)")

# Pose: whole recording, as an image (rows = joints, colour = angle)
t_pose = np.arange(len(pose_deg)) / fs
im = ax[1].imshow(pose_deg.T, aspect="auto", cmap="RdBu_r", vmin=-90, vmax=90,
                  extent=[0, t_pose[-1], 19.5, -0.5], interpolation="nearest")
ax[1].set_yticks(range(20), JOINT_NAMES, fontsize=7)
ax[1].set_xlabel("time (s)"); ax[1].set_title(f"Joint angles, whole recording ({stage})")
plt.colorbar(im, ax=ax[1], label="angle (°)")
plt.tight_layout(); plt.show()

In [ ]:
# 📊 Zoom in: one finger's joints over 10 s, next to one EMG channel
sl = slice(int(10 * fs), int(20 * fs))
tt = np.arange(len(emg_raw))[sl] / fs
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(tt, emg_raw[sl, 0], lw=0.3, color="k"); ax[0].set_ylabel("raw EMG ch0")
for j in [5, 6, 7]:
    ax[1].plot(tt, pose_deg[sl, j], label=JOINT_NAMES[j])
ax[1].set_ylabel("angle (°)"); ax[1].set_xlabel("time (s)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

✍️ **Q1.** Look at the two plots. How is raw EMG different from the joint angles (speed, sign, smoothness)?
Do any stretches of the pose look suspicious, e.g. every joint sitting at exactly 0°?

> *Your answer:*

### 📊 All 30 recordings at a glance
This only reads labels and lengths, so it is fast. Note the **two-handed** stages are much longer, and every stage
appears once per hand.

In [ ]:
info = []
for p in files:
    with h5py.File(p, "r") as f:
        g = f["emg2pose"]
        info.append((str(g.attrs["stage"]), str(g.attrs["side"]), len(g["timeseries"]) / g.attrs["sample_rate"]))

fig, ax = plt.subplots(figsize=(10, 7))
labels = [f"{s} ({h[0].upper()})" for s, h, _ in info]
colors = ["C0" if h == "left" else "C1" for _, h, _ in info]
ax.barh(labels, [d for *_, d in info], color=colors)
ax.invert_yaxis(); ax.set_xlabel("duration (s)")
ax.set_title("30 recordings: blue = left hand, orange = right hand")
plt.tight_layout(); plt.show()
print(f"{len(set(s for s, _, _ in info))} stages · total {sum(d for *_, d in info)/60:.1f} min")

---
## Part 2 · From raw EMG to an envelope

Raw EMG is a fast, noisy signal that swings positive and negative. What we care about is **how active** each
muscle area is, which is the signal's *size*, changing slowly. Getting there takes five steps:

| Step | What | Why |
|---|---|---|
| 1 | **Band-pass 20–450 Hz** | Muscle signal lives here. Below 20 Hz is movement drift; above 450 Hz is mostly noise |
| 2 | **Notch at 60 Hz** | Remove mains electricity hum |
| 3 | **Rectify** (absolute value) | Size is what matters, not sign |
| 4 | **Low-pass at 10 Hz** | Smooth the rectified signal into an envelope |
| 5 | **Downsample to 50 Hz, then scale** | The smoothing removed everything faster than 10 Hz, so fewer samples lose nothing. Scaling evens out electrode contact between recordings |

### 📊 First, where is the energy? (power spectrum)
A power spectrum shows how much of the signal sits at each frequency. Steps 1–2 are decided by this plot.

In [ ]:
freqs, psd = signal.welch(emg_raw, fs=fs, nperseg=2048, axis=0)   # psd: (n_freqs, 16)
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs, psd.mean(axis=1), color="k")
ax.axvspan(20, 450, color="C2", alpha=0.15, label="kept by band-pass (20–450 Hz)")
ax.axvline(60, color="C3", ls="--", label="60 Hz mains")
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("power (log scale)"); ax.set_xlim(0, 1000)
ax.set_title("Raw EMG power spectrum, mean over 16 channels"); ax.legend()
plt.tight_layout(); plt.show()

### Task 2a · Band-pass and notch filter
Use **zero-phase** filtering (`filtfilt`): it runs the filter forwards then backwards, so the output is not shifted in
time. A time shift would misalign EMG with pose.

`scipy.signal` cheatsheet:
- `sos = signal.butter(4, [low, high], btype="bandpass", fs=fs, output="sos")` designs a filter
- `signal.sosfiltfilt(sos, x, axis=0)` applies it along time (axis 0), separately for each channel
- `b, a = signal.iirnotch(60.0, Q=30.0, fs=fs)` then `signal.filtfilt(b, a, x, axis=0)` for the notch

In [ ]:
def bandpass_notch(emg_raw: np.ndarray, fs: float) -> np.ndarray:
    """Steps 1–2: 4th-order Butterworth band-pass 20–450 Hz, then a 60 Hz notch (Q=30), both zero-phase.

    emg_raw : (T, 16) raw EMG
    returns : (T, 16) filtered EMG, same shape, still positive AND negative
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2a? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Two filters, one after the other, each in two steps: **design** it (get its coefficients), then **apply** it to
the data along the time axis.
1. Design a band-pass that keeps 20–450 Hz, apply it to `emg_raw`.
2. Design a 60 Hz notch, apply it to the result of step 1.
3. Return the result. The shape never changes: filtering doesn't add or remove samples.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `signal.butter(order, cutoff, btype=..., fs=fs, output="sos")` designs a Butterworth filter. Order 4 is a
  standard "steepness". Give `fs=fs` and you can write cut-offs in Hz. For a band-pass, `cutoff` is a pair `[20, 450]`.
- Filters come in two formats, and each has its own apply function. Match them:
  `sos` → `signal.sosfiltfilt(sos, x, axis=0)`; `(b, a)` → `signal.filtfilt(b, a, x, axis=0)`.
- `axis=0` = filter down the rows (along time), so each of the 16 columns is filtered separately.

```python
# Toy: a slow 5 Hz wave plus a fast 100 Hz wave. The band-pass should remove the 5 Hz part.
t = np.arange(2000) / 2000
x = (np.sin(2*np.pi*5*t) + np.sin(2*np.pi*100*t))[:, None]   # (2000, 1): 1 channel
sos = signal.butter(4, [20, 450], btype="bandpass", fs=2000, output="sos")
y = signal.sosfiltfilt(sos, x, axis=0)
plt.plot(t, x, alpha=0.4, label="before"); plt.plot(t, y, label="after"); plt.legend(); plt.xlim(0, 0.5)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def bandpass_notch(emg_raw: np.ndarray, fs: float) -> np.ndarray:
    sos_bp = signal.butter(4, [____, ____], btype="bandpass", fs=fs, output="sos")
    x = signal.sosfiltfilt(sos_bp, emg_raw, axis=____)
    b, a = signal.iirnotch(____, Q=30.0, fs=fs)
    return signal.filtfilt(b, a, ____, axis=0)
```

</details>

In [ ]:
# ✅ Check Task 2a
emg_filt = bandpass_notch(emg_raw, fs)
assert emg_filt.shape == emg_raw.shape, f"shape {emg_filt.shape}, expected {emg_raw.shape}"
f2, psd_filt = signal.welch(emg_filt, fs=fs, nperseg=2048, axis=0)
low, hum, mid = psd_filt[f2 < 5].mean(), psd_filt[np.argmin(abs(f2 - 60))].mean(), psd_filt[(f2 > 80) & (f2 < 150)].mean()
assert low < 0.01 * mid, "Too much power left below 5 Hz: is the band-pass applied?"
assert hum < mid, "60 Hz is still strong: is the notch applied?"
print("✅ band-pass and notch look right")

# 📊 Spectrum before and after
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(freqs, psd.mean(axis=1), color="0.6", label="raw")
ax.semilogy(f2, psd_filt.mean(axis=1), color="C0", label="band-pass + notch")
ax.set_xlim(0, 1000); ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("power"); ax.legend()
plt.tight_layout(); plt.show()

### Task 2b · Rectify and smooth
Take the absolute value, then low-pass at 10 Hz (4th-order Butterworth, zero-phase). Filtering can dip a hair below
zero, so finish with `np.clip(env, 1e-8, None)`: NMF later needs **every value ≥ 0**.

In [ ]:
def rectify_and_smooth(x: np.ndarray, fs: float) -> np.ndarray:
    """Steps 3–4: |x|, then a 10 Hz low-pass (zero-phase), then clip to ≥ 1e-8.

    x       : (T, 16) band-passed EMG from Task 2a
    returns : (T, 16) envelope, still at the original sample rate, all values > 0
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2b? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Three lines:
1. Rectify: make every value positive (flip the negative half up).
2. Smooth: low-pass the rectified signal at 10 Hz, same design-then-apply pattern as Task 2a.
3. Clip: raise anything below 1e-8 up to 1e-8, so nothing is zero or negative.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.abs(x)` absolute value, element by element.
- A low-pass has a single cut-off, not a pair: `signal.butter(4, 10, btype="lowpass", fs=fs, output="sos")`.
- `np.clip(a, low, None)` sets a floor and no ceiling (`None` = no upper limit).

```python
a = np.array([-3.0, -0.5, 0.0, 2.0])
np.abs(a)                 # array([3. , 0.5, 0. , 2. ])
np.clip(a, 1e-8, None)    # array([1.e-08, 1.e-08, 1.e-08, 2.e+00])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def rectify_and_smooth(x: np.ndarray, fs: float) -> np.ndarray:
    sos_lp = signal.butter(4, ____, btype="lowpass", fs=fs, output="sos")
    env = signal.sosfiltfilt(sos_lp, ____, axis=0)     # filter the RECTIFIED signal
    return np.clip(env, ____, None)
```

</details>

In [ ]:
# ✅ Check Task 2b
env_full = rectify_and_smooth(emg_filt, fs)
assert env_full.shape == emg_raw.shape, f"shape {env_full.shape}, expected {emg_raw.shape}"
assert (env_full > 0).all(), "envelope has values ≤ 0: did you rectify and clip?"
roughness = np.abs(np.diff(env_full[:, 0])).mean() / env_full[:, 0].mean()
assert roughness < 0.01, "envelope is still jagged: is the 10 Hz low-pass applied to |x|?"
print("✅ envelope is positive and smooth")

# 📊 The four stages, 2 s of channel 0
n = int(2 * fs); t = np.arange(n) / fs; ch = 0
fig, ax = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
ax[0].plot(t, emg_raw[:n, ch], lw=0.4, color="0.4");   ax[0].set_ylabel("1. raw")
ax[1].plot(t, emg_filt[:n, ch], lw=0.4, color="C0");   ax[1].set_ylabel("2. filtered")
ax[2].plot(t, np.abs(emg_filt[:n, ch]), lw=0.4, color="C1"); ax[2].set_ylabel("3. rectified")
ax[3].plot(t, np.abs(emg_filt[:n, ch]), lw=0.3, color="C1", alpha=0.3)
ax[3].plot(t, env_full[:n, ch], lw=2, color="C3");     ax[3].set_ylabel("4. envelope")
ax[3].set_xlabel("time (s)"); fig.suptitle(f"Raw EMG → envelope, channel {ch}")
plt.tight_layout(); plt.show()

### Task 2c · Downsample and scale
- **Downsample:** keep every `fs / fs_out`-th row (2000 / 50 = 40). Slicing does it: `env[::step]`.
- **Scale:** divide by the 95th percentile of the whole recording (all channels together), so a typical "high"
  value is about 1. Add `1e-8` to the divisor so you never divide by zero. Use the 95th percentile, not the max,
  so one spike can't squash everything.

Return `float32` to keep the file small.

In [ ]:
def downsample_and_scale(env_full: np.ndarray, fs: float, fs_out: int = FS_OUT) -> np.ndarray:
    """Step 5.

    env_full : (T, 16) envelope at fs
    returns  : (T // step, 16) float32, step = round(fs / fs_out), scaled so its 95th percentile ≈ 1
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2c? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Work out the step: how many raw samples per output sample? 2000 / 50 = 40. Use `int(round(...))`, because
   slicing needs a whole number.
2. Keep every step-th row.
3. Find the 95th percentile of *all* values in that downsampled array (one number).
4. Divide by it (plus 1e-8), and convert to float32.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `x[::step]` keeps rows 0, step, 2·step, … (all columns).
- `np.percentile(x, 95)` with no `axis` treats the whole 2-D array as one long list and returns a single number.
- `.astype(np.float32)` at the end.

```python
x = np.arange(10, dtype=float)[:, None]   # (10, 1)
x[::3].ravel()                            # array([0., 3., 6., 9.])
v = np.r_[np.arange(1, 101), 10_000]      # 100 ordinary values + 1 huge spike
np.percentile(v, 95), v.max()             # (96.0, 10000): the spike barely moves the 95th percentile
int(round(2000 / 50))                     # 40
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def downsample_and_scale(env_full: np.ndarray, fs: float, fs_out: int = FS_OUT) -> np.ndarray:
    step = int(round(____ / ____))
    env = env_full[::____]
    scale = np.percentile(env, ____) + 1e-8
    return (env / scale).astype(____)
```

</details>

In [ ]:
# ✅ Check Task 2c: your whole envelope pipeline against the reference
def to_envelope(emg_raw: np.ndarray, fs: float) -> np.ndarray:
    return downsample_and_scale(rectify_and_smooth(bandpass_notch(emg_raw, fs), fs), fs)

env = to_envelope(emg_raw, fs)
env_ref = ref.to_envelope(emg_raw, fs)
assert env.shape == env_ref.shape, f"shape {env.shape}, expected {env_ref.shape}"
assert env.dtype == np.float32, f"dtype {env.dtype}, expected float32"
assert abs(np.percentile(env, 95) - 1) < 1e-3, "95th percentile should be 1 after scaling"
assert np.allclose(env, env_ref, rtol=1e-3, atol=1e-5), \
    f"values differ from the reference (max diff {np.abs(env - env_ref).max():.2e})"
print(f"✅ matches the reference: {emg_raw.shape} raw → {env.shape} envelope "
      f"({emg_raw.size / env.size:.0f}× fewer numbers)")

### 📊 The whole recording: EMG envelope next to pose
Now both are at 50 samples/s, so they line up row for row. Look for moments where the EMG lights up and the pose
changes. That link is what the whole project is about.

In [ ]:
pose_50 = pose_deg[:: int(round(fs / FS_OUT))][: len(env)]
t50 = np.arange(len(env)) / FS_OUT
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
im0 = ax[0].imshow(env.T, aspect="auto", cmap="magma", vmin=0, vmax=1.5,
                   extent=[0, t50[-1], 15.5, -0.5], interpolation="nearest")
ax[0].set_ylabel("EMG channel"); ax[0].set_title(f"EMG envelope ({stage}, {side} hand)")
plt.colorbar(im0, ax=ax[0], label="scaled activity")
im1 = ax[1].imshow(pose_50.T, aspect="auto", cmap="RdBu_r", vmin=-90, vmax=90,
                   extent=[0, t50[-1], 19.5, -0.5], interpolation="nearest")
ax[1].set_yticks(range(20), JOINT_NAMES, fontsize=6); ax[1].set_xlabel("time (s)")
ax[1].set_title("Joint angles"); plt.colorbar(im1, ax=ax[1], label="angle (°)")
plt.tight_layout(); plt.show()

✍️ **Q2.** In one or two sentences each: why zero-phase filtering? Why is it safe to downsample to 50 Hz *after*
the 10 Hz low-pass but not before? Why scale each recording separately?

> *Your answer:*

---
## Part 3 · Bad pose frames

The pose comes from motion capture. When the cameras lose the hand, **all 20 angles are stored as exactly 0**
(not NaN). If we trained on those frames we would be teaching a model that the hand sometimes snaps to a flat
"all zeros" pose. So we mark them with a mask, `pose_valid`, and drop them from anything that uses pose.
EMG-only analyses (like NMF) can still use those frames.

### Task 3 · A mask of valid pose frames

In [ ]:
def valid_pose_mask(pose: np.ndarray) -> np.ndarray:
    """pose : (T, 20) degrees  →  (T,) bool, True where the frame is usable.

    A frame is INVALID if all 20 angles are 0 (use np.isclose(pose, 0.0), not == 0)
    or if any angle is NaN. Hint: np.all(..., axis=1) and np.isnan(...).any(axis=1).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

A frame (row) is bad in two cases. Build one True/False per row for each case, then combine:
1. `all_zero`: every one of the 20 angles in that row is (close to) 0.
2. `has_nan`: at least one angle in that row is NaN.
3. Valid = **not** all_zero **and not** has_nan.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.isclose(pose, 0.0)` → same-shape array of True/False ("is this value ≈ 0?").
- `.all(axis=1)` / `np.all(..., axis=1)`: is **every** value across the row True? → one answer per row, shape (T,).
- `np.isnan(pose).any(axis=1)`: is **any** value in the row NaN? → (T,).
- `~a` flips True/False; `a & b` is True only where both are.

```python
p = np.array([[0.0, 0.0], [0.0, 5.0], [np.nan, 1.0]])
np.isclose(p, 0.0).all(axis=1)     # array([ True, False, False])
np.isnan(p).any(axis=1)            # array([False, False,  True])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def valid_pose_mask(pose: np.ndarray) -> np.ndarray:
    all_zero = np.all(np.isclose(pose, 0.0), axis=____)
    has_nan = np.isnan(pose).any(axis=____)
    return ~all_zero & ____
```

</details>

In [ ]:
# ✅ Check Task 3
toy = np.array([[0.0] * 20, [1.0] + [0.0] * 19, [np.nan] + [5.0] * 19, [10.0] * 20])
assert valid_pose_mask(toy).tolist() == [False, True, False, True], \
    f"toy test: got {valid_pose_mask(toy).tolist()}, expected [False, True, False, True]"
valid = valid_pose_mask(pose_50)
assert valid.dtype == bool and valid.shape == (len(pose_50),)
assert (valid == ref.valid_pose_mask(pose_50)).all()
print(f"✅ {valid.mean():.1%} of frames in this recording have valid pose")

# 📊 Where do the failures happen?
fig, ax = plt.subplots(figsize=(14, 3))
for j in [5, 9, 13, 17]:
    ax.plot(t50, pose_50[:, j], lw=0.7, label=JOINT_NAMES[j])
ax.legend(fontsize=7, loc="upper right")
for start, stop in zip(*[np.flatnonzero(np.diff(np.r_[0, ~valid, 0].astype(int)) == s) for s in (1, -1)]):
    ax.axvspan(start / FS_OUT, stop / FS_OUT, color="red", alpha=0.25)
ax.set_xlabel("time (s)"); ax.set_ylabel("MCP flexion (°)")
ax.set_title("Knuckle angles; red = tracking failed (pose_valid = False)")
plt.tight_layout(); plt.show()

---
## Part 4 · Train/test split, in time

We test on **new moments of the same tasks**: the first 80% of each recording is train, the last 20% is test.

Why not shuffle frames randomly? Neighbouring frames are almost identical (and the 10 Hz smoothing blends them),
so a random split would put near-copies of test frames in train. Scores would look great and mean nothing.
The **1 s gap** between train and test makes sure no smoothing window touches both.

### Task 4 · Split labels

In [ ]:
def split_labels(n: int, train_fraction: float = TRAIN_FRACTION, gap: int = GAP) -> np.ndarray:
    """(n,) int8 labels for one recording of n frames.

    0 for the first int(train_fraction * n) frames (train),
    -1 for the next `gap` frames (gap: used by neither),
    1 for everything after (test).
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Start with `n` zeros (everything is train).
2. Find where train ends: `cut = int(train_fraction * n)`.
3. Overwrite the `gap` rows starting at `cut` with -1.
4. Overwrite everything after the gap with 1.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.zeros(n, dtype=np.int8)` makes the starting array (int8 = small whole numbers, saves memory).
- Assigning to a slice changes that part in place: `lab[a:b] = -1`. `lab[b:] = 1` means "from b to the end".
- Slicing past the end is fine in NumPy; it just stops at the end.

```python
lab = np.zeros(10, dtype=np.int8)
lab[8:9] = -1
lab[9:] = 1
lab          # array([ 0,  0,  0,  0,  0,  0,  0,  0, -1,  1], dtype=int8)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def split_labels(n: int, train_fraction: float = TRAIN_FRACTION, gap: int = GAP) -> np.ndarray:
    lab = np.zeros(n, dtype=np.int8)
    cut = int(____ * n)
    lab[cut: cut + ____] = -1
    lab[____:] = 1
    return lab
```

</details>

In [ ]:
# ✅ Check Task 4
assert split_labels(10, 0.8, 1).tolist() == [0] * 8 + [-1] + [1], \
    f"split_labels(10, 0.8, 1) = {split_labels(10, 0.8, 1).tolist()}"
lab = split_labels(len(env))
assert lab.dtype == np.int8 and (lab == ref.split_labels(len(env))).all()
print(f"✅ train {np.sum(lab == 0)} · gap {np.sum(lab == -1)} · test {np.sum(lab == 1)} frames")

# 📊 The split on the timeline
fig, ax = plt.subplots(figsize=(14, 2.5))
ax.plot(t50, env.mean(axis=1), color="k", lw=0.6)
for value, color, name in [(0, "C0", "train"), (-1, "0.5", "gap"), (1, "C1", "test")]:
    ax.fill_between(t50, 0, env.mean(axis=1).max(), where=lab == value, color=color, alpha=0.25, label=name)
ax.set_xlabel("time (s)"); ax.set_ylabel("mean envelope"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

---
## Part 5 · Run it on all 30 recordings

Now put your functions together. For each recording: load it, make the envelope, downsample the pose to the same
50 Hz, mark valid frames, make split labels. Then stack all recordings into long arrays, adding `rec` (which
recording each row came from) so nothing gets lost.

### Task 5 · Line the pose up with the envelope
The envelope was downsampled by taking every 40th row. The pose must be downsampled **the same way**, and then
cut to the same length, so row *i* of the EMG and row *i* of the pose are the same moment.

In [ ]:
cols: dict[str, list[np.ndarray]] = {k: [] for k in ["emg", "pose", "pose_valid", "rec", "split"]}
stages, sides, names = [], [], []

for r, path in enumerate(files):
    emg_raw, pose_deg, fs, stage, side = load_recording(path)
    env = to_envelope(emg_raw, fs)

    p = ...      # TODO: (len(env), 20) pose at 50 Hz: every round(fs / FS_OUT)-th row, cut to len(env)
    env = env[: len(p)]

    for k, v in zip(cols, [env, p.astype(np.float32), valid_pose_mask(p),
                           np.full(len(env), r, np.int16), split_labels(len(env))]):
        cols[k].append(v)
    stages.append(stage); sides.append(side); names.append(Path(path).stem)
    print(f"{r + 1:2d}/{len(files)}  {stage} ({side})")

data = {k: np.concatenate(v) for k, v in cols.items()}
data.update(stage=np.array(stages), side=np.array(sides), filename=np.array(names), fs=np.array(FS_OUT))
print(f"Built {len(data['emg']):,} frames")

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

The envelope kept every 40th raw sample. Do **exactly the same** to the pose, then trim it to the envelope's
length, so row i of each is the same moment in time. Two slices, one after the other.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- You can chain slices: `a[::40][:n]` first keeps every 40th row, then keeps the first n of those.
- The step is the same one you used in Task 2c: `int(round(fs / FS_OUT))`.

```python
a = np.arange(200)[:, None]    # 200 "raw" rows
a[::40][:3].ravel()            # array([ 0, 40, 80])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
p = pose_deg[:: int(round(____ / ____))][: len(____)]
```

</details>

In [ ]:
# ✅ Final check: your dataset against the team's reference file
import subprocess
if not PROCESSED.exists():
    subprocess.run([sys.executable, str(PREP_DIR / "prep.py")], check=True)
d_ref = np.load(PROCESSED)

for k in ["emg", "pose", "pose_valid", "rec", "split"]:
    assert data[k].shape == d_ref[k].shape, f"{k}: shape {data[k].shape}, reference {d_ref[k].shape}"
    assert data[k].dtype == d_ref[k].dtype, f"{k}: dtype {data[k].dtype}, reference {d_ref[k].dtype}"
assert np.allclose(data["emg"], d_ref["emg"], rtol=1e-3, atol=1e-5), "emg differs from the reference"
for k in ["pose", "pose_valid", "rec", "split"]:
    assert np.array_equal(data[k], d_ref[k]), f"{k} differs from the reference"
assert (data["emg"] >= 0).all(), "envelope must be non-negative for NMF"
print("✅ Your dataset matches the reference. Notebooks 02–06 load the same data.")

---
## Part 6 · What is in `processed_mini.npz`?

This is the **format every other notebook uses**. One row = one frame = 20 ms. All recordings are stacked one after
another, so `N` ≈ 90,000 rows. Per-frame arrays have N rows; per-recording arrays have 30.

| Key | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float32 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float32 | joint angles in **degrees**, columns in `JOINT_NAMES` order |
| `pose_valid` | (N,) | bool | False where pose tracking failed |
| `rec` | (N,) | int16 | which recording (0–29) the row comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap |
| `stage` | (30,) | str | task name per recording; per frame: `stage[rec]` |
| `side` | (30,) | str | `"left"` / `"right"` per recording; per frame: `side[rec]` |
| `filename` | (30,) | str | original file name per recording |
| `fs` | () | int | 50, samples per second |

Boolean arrays are **masks**: `emg[d["split"] == 0]` keeps the train rows, `pose[d["pose_valid"]]` keeps good pose.
The plots below explore it.

In [ ]:
d = np.load(PROCESSED)
emg, pose, rec, split, pose_valid = d["emg"], d["pose"], d["rec"], d["split"], d["pose_valid"]
stage_of_frame = d["stage"][rec]          # (N,) per-recording label → per-frame label
for k in d.files:
    print(f"{k:11s} {str(d[k].shape):12s} {d[k].dtype}")

# 📊 Per recording: how many frames, and how much pose is valid
n_frames = np.bincount(rec)
valid_pct = np.array([pose_valid[rec == r].mean() for r in range(len(d["stage"]))]) * 100
labels = [f"{s} ({h[0].upper()})" for s, h in zip(d["stage"], d["side"])]
fig, ax = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
ax[0].barh(labels, n_frames / FS_OUT); ax[0].set_xlabel("duration (s)"); ax[0].invert_yaxis()
ax[1].barh(labels, valid_pct, color=["C3" if v < 90 else "C2" for v in valid_pct])
ax[1].axvline(90, color="k", ls=":"); ax[1].set_xlabel("% frames with valid pose")
plt.tight_layout(); plt.show()

In [ ]:
# 📊 What values do EMG and pose take?
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(emg.ravel(), bins=200, range=(0, 3), color="C3")
ax[0].set_xlabel("envelope value"); ax[0].set_ylabel("count"); ax[0].set_yscale("log")
ax[0].set_title("EMG envelope: non-negative, mostly small, long tail")
ax[1].boxplot(pose[pose_valid], vert=False, showfliers=False)
ax[1].set_yticks(range(1, 21), JOINT_NAMES, fontsize=7); ax[1].invert_yaxis()
ax[1].set_xlabel("angle (°)"); ax[1].set_title("Range of each joint (valid frames)")
plt.tight_layout(); plt.show()

In [ ]:
# 📊 Which stages use which channels? Mean envelope per stage × channel
stages_u = sorted(set(d["stage"]))
mean_env = np.array([emg[stage_of_frame == s].mean(axis=0) for s in stages_u])
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(mean_env, aspect="auto", cmap="magma")
ax.set_yticks(range(len(stages_u)), stages_u, fontsize=8); ax.set_xticks(range(16))
ax.set_xlabel("EMG channel"); ax.set_title("Mean envelope per stage (both hands)")
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

### 📊 A preview of why synergies and eigenpostures might work
If channels (or joints) rise and fall **together**, a few patterns can describe all of them. That is the bet behind
NMF synergies (notebook 02) and PCA eigenpostures. Below: how strongly each pair moves together (correlation).
Big bright blocks mean shared structure.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
im0 = ax[0].imshow(np.corrcoef(emg[split == 0].T), cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set_title("EMG channel × channel correlation (train)"); ax[0].set_xticks(range(16)); ax[0].set_yticks(range(16))
im1 = ax[1].imshow(np.corrcoef(pose[(split == 0) & pose_valid].T), cmap="RdBu_r", vmin=-1, vmax=1)
ax[1].set_title("Joint × joint correlation (train, valid pose)")
ax[1].set_yticks(range(20), JOINT_NAMES, fontsize=6); ax[1].set_xticks([])
plt.colorbar(im1, ax=ax, shrink=0.8); plt.show()

✍️ **Q3.** What is one row of `processed_mini.npz`? How would you get only the test frames of the right hand
with valid pose? (Write the one line of NumPy.)

✍️ **Q4.** From the correlation plots: do the EMG channels look like they share structure? Do the joints? Which
side do you expect to compress into fewer dimensions, and why?

> *Your answers:*

---
**Before you close:** add a short entry to [`log.md`](log.md): what you learned, what confused you, what you'd
check next.

---
## Part 7 · Methodology review: what holds up, what to question

This part steps back from *how* to code each step and asks whether the choices are *right*, using evidence from
the data. Read 7.1 first if the settings at the top of the notebook (`FS_OUT`, `TRAIN_FRACTION`, `GAP`) still
feel abstract. Then run the evidence cell and read 7.2–7.4 next to its output.

### 7.1 · The three settings, in plain words

| Setting | Value | Unit | What it controls |
|---|---|---|---|
| `FS_OUT` | 50 | frames per second | How often we keep a sample. The raw data has 2,000 per second, so we keep **1 in 40**. One frame = 20 ms. |
| `TRAIN_FRACTION` | 0.8 | share of each recording | The **first 80%** of every recording (in time) is train. |
| `GAP` | `FS_OUT` = 50 | **frames**, not seconds | 50 frames × 20 ms = **1 s** thrown away between train and test. Written as `FS_OUT` so it stays 1 s if you change the frame rate. |

A 60 s recording, after downsampling, is 60 × 50 = 3,000 frames. It gets split like this:

```
frame:  0                                      2400   2450                3000
        |---------------- train (label 0) -------|-gap-|---- test (label 1) ---|
        |                 48 s                   | 1 s |         11 s          |
                                                   -1
```

**Why throw away 1 s?** The smoothing filter (Task 2b) works like a sliding average: each envelope value mixes in
raw samples from about **±0.25 s around it, both earlier and later** (that is what zero-phase / `filtfilt` means).
Without a gap, the last train frames would contain a little of the first test frames, so the model would have
"seen" part of the test. A 1 s gap is four times wider than the filter's reach, so nothing crosses it.

**Why 50 frames per second is enough:** after the 10 Hz smoothing, the envelope has no detail faster than 10 Hz.
To capture a signal you need at least twice its fastest frequency (the *Nyquist* rule), here 20 per second. 50 per
second leaves a safe margin, and keeping more frames would only repeat near-identical values.

In [ ]:
# 📊 Evidence for the methodology review: just run this and read the output.
# Uses `files`, `ref`, `FS_OUT`, `GAP` from the top of the notebook and `emg, pose, rec, split, pose_valid` from Part 6.

# A. What the raw files say about who, what and how big
raw_size, attrs = [], []
for p in files:
    with h5py.File(p, "r") as f:
        g = f["emg2pose"]
        t = g["timeseries"]["time"][:]
        raw_size.append(np.percentile(np.abs(g["timeseries"]["emg"][::40]), 95))
        attrs.append({k: str(g.attrs[k]) for k in ["stage", "side", "moving_hand", "user", "held_out_stage"]})
        attrs[-1]["time_jumps"] = int((np.diff(t) > 1.5 * np.median(np.diff(t))).sum())
raw_size = np.array(raw_size)
print(f"A1. Users in the mini dataset: {sorted({a['user'] for a in attrs})}")
print(f"A2. Recordings with dropped samples (time jumps): {sum(a['time_jumps'] > 0 for a in attrs)} of {len(files)}")
print(f"A3. Raw EMG size before our scaling (95th pct of |EMG|): {raw_size.min():.1f} to {raw_size.max():.1f}, "
      f"a {raw_size.max() / raw_size.min():.0f}x range")
idle = [f"{a['stage']} ({a['side']})" for a in attrs if a["moving_hand"] not in ("both", a["side"])]
print(f"A4. Recordings where this hand is NOT the one doing the task: {idle}")
print(f"A5. Stages the dataset itself marks as held-out: {sorted({a['stage'] for a in attrs if a['held_out_stage'] == 'True'})}")

# B. Is the test part like the train part?
def pose_shift(r: int) -> float:
    """Mean |train mean − test mean| over joints, in units of the train standard deviation."""
    a = pose[(rec == r) & (split == 0) & pose_valid]
    b = pose[(rec == r) & (split == 1) & pose_valid]
    return float(np.abs(a.mean(0) - b.mean(0)).mean() / a.std(0).mean()) if len(a) and len(b) else np.nan
shift = np.array([pose_shift(r) for r in range(len(files))])
print(f"B1. Test pose differs from train pose by a median {np.nanmedian(shift):.2f} train-SDs (max {np.nanmax(shift):.2f})")
print(f"B2. Shortest test block: {min(np.sum((rec == r) & (split == 1)) for r in range(len(files))) / FS_OUT:.0f} s")

# C. Does EMG lead the movement?
def best_lag(r: int, max_lag: int = 15) -> int:
    """Lag (frames) that best lines up mean EMG with pose speed. Positive = EMG comes first."""
    m = rec == r
    e = emg[m].mean(axis=1)
    speed = np.r_[0, np.linalg.norm(np.diff(pose[m], axis=0), axis=1)]
    ok = pose_valid[m] & np.r_[False, pose_valid[m][:-1]]          # this frame and the previous are both valid
    lags = np.arange(-max_lag, max_lag + 1)
    cc = [np.corrcoef(np.roll(e, L)[ok][max_lag:-max_lag], speed[ok][max_lag:-max_lag])[0, 1] for L in lags]
    return int(lags[np.argmax(cc)])
lag = np.array([best_lag(r) for r, a in enumerate(attrs) if a["moving_hand"] in ("both", a["side"])])
print(f"C1. Best EMG→pose lag over moving-hand recordings: median {np.median(lag):.0f} frames "
      f"= {np.median(lag) / FS_OUT * 1000:.0f} ms (EMG first)")

# D. Was the EMG already filtered before we got it? Is the gap long enough?
x, fs0 = ref.load_recording(files[0])[0], ref.load_recording(files[0])[2]
fr, P = signal.welch(x, fs=fs0, nperseg=8192, axis=0)
P = P.mean(axis=1)
at = lambda hz: P[np.argmin(np.abs(fr - hz))]
print(f"D1. Share of raw EMG power below 20 Hz: {P[fr < 20].sum() / P.sum():.2%}")
print(f"D2. Power at 60 Hz = {at(60):.1f} vs 58 Hz = {at(58):.1f} and 62 Hz = {at(62):.1f}: a dip, not a hum peak")
sos = signal.butter(4, 10, fs=fs0, output="sos")
imp = np.zeros(int(4 * fs0)); imp[len(imp) // 2] = 1
h = np.abs(signal.sosfiltfilt(sos, imp))
print(f"D3. The 10 Hz smoother mixes samples up to ±{np.ptp(np.flatnonzero(h > 1e-3 * h.max())) / 2 / fs0 * 1000:.0f} ms "
      f"apart; the gap is {GAP / FS_OUT * 1000:.0f} ms")

### 7.2 · What holds up ✅

| Choice | Verdict | Evidence |
|---|---|---|
| Zero-phase filtering | Right | Keeps EMG and pose lined up in time; an ordinary filter would delay the EMG. |
| 10 Hz smoothing, then 50 Hz | Right | Nothing above 25 Hz survives the smoother, so keeping 1 in 40 samples loses nothing. |
| Downsampling pose by slicing (no filter) | Fine | In the first recording only ~0.6% of pose power is above 25 Hz: the hand moves slowly, so there is almost nothing to alias. |
| Slicing every 40th row = equal time steps | Fine | **A2**: no recording has dropped samples, so row *i* really is *i* × 20 ms. |
| Split in time, not random | Right | Neighbouring frames are near-copies; a random split would leak them into the test set. |
| 1 s gap | Right, and generous | **D3**: the smoother reaches ±~250 ms; the gap is 1,000 ms. |
| 95th percentile instead of max | Right | One spike can't shrink the whole recording. |
| Clipping to ≥ 1e-8 | Right | NMF needs non-negative input. |
| Masking all-zero pose frames | Right, but see 7.3 (6) | Keeps "tracking lost" from being learned as a real hand pose. |

### 7.3 · What to question, most important first ⚠️

**1. Per-recording scaling makes idle hands look busy, and removes effort differences.**
**A3** shows the raw EMG size varies **50×** between recordings. The two smallest are the hands that were *not*
doing the task (**A4**: `unconstrained`, where the other hand moves). Scaling each recording so its 95th percentile
is 1 multiplies their resting noise by up to ~50× until it looks like real activity. Among working hands the range
is still about 5× (27 to 127), which probably reflects real effort, since it is one person in one session with the
same armband. After scaling, that information is gone. So the Part 6 "mean envelope per stage" plot compares
*patterns* across channels, not *how hard* each stage was. The scale is also computed over the whole recording,
test 20% included: a small leak.
*Suggestions:* (a) compute the scale from **train frames only**; (b) consider **one scale per hand (armband) for the
whole session** instead of one per recording, since contact doesn't change between recordings; (c) store
`moving_hand` in the `.npz` and drop or flag idle-hand recordings.

**2. The test set is "the end of the task", which is not the same as "more of the task".**
**B1**: on average the test pose sits 0.57 train-standard-deviations away from the train pose (up to 1.38). Many
stages are sequences (counting up *then* down), so the last 20% can contain movements that never appear in train.
A low test score could then mean "train never showed this", not "the model fails". With one split per recording
and test blocks as short as **8 s** (**B2**), one awkward ending can swing the result.
*Suggestion:* **blocked cross-validation in time.** Cut each recording into 5 blocks; each block takes one turn as
test, with a 1 s gap on both sides; report the mean and spread over the 5 folds.

**3. Decide what "generalise" means, and pick the split to match.**
The current split asks: *new moments of tasks the model has seen.* Two harder questions may matter more for the
synergy claims: *new movements* (**A5**: the dataset already marks 4 stages as held-out stages) and *new people*.
*Suggestion:* add a leave-stage-out split. If synergies are "real" building blocks, they should hold up on stages
never used to fit them.

**4. Only one person.**
**A1**: all 30 recordings are from one user in one session. Every result is a claim about this one person's hand.
That is fine for a pilot, but the write-up should say so. Cross-user claims need more users from the full dataset.

**5. EMG comes before movement, but we line them up at the same instant.**
Muscles fire, then the fingers move a little later (electromechanical delay). **C1** is a rough check: average EMG
lines up best with pose *speed* when EMG is shifted ~**3 frames (≈ 60 ms) earlier**. Row *i* of `emg` and row *i*
of `pose` are the same clock time, which is correct for storage. But a model that predicts pose from the *same*
frame of EMG is asking slightly the wrong question.
*Suggestion:* keep the file as it is, and in notebook 04 try predicting `pose[t]` from `emg[t - k]` for k = 0…10,
or from a short window of past EMG. Report the best k.

**6. "Valid pose" only catches total failures.**
The mask finds frames where all 20 angles are 0 (~16% of frames). Tracking can also fail *partly*: one finger
jumps, or the frames just before and after a dropout are wobbly.
*Suggestion:* also flag frames where the pose jumps faster than a hand can move, and pad each dropout by a few
frames.

**7. Two filter steps repeat work already done on the data (low priority).**
**D1–D2**: the shipped EMG has almost no power below 20 Hz, and at 60 Hz it shows a **dip, not a hum peak**. The data
was already high-pass and notch filtered before release. Our band-pass and notch don't hurt (the notch removes a
sliver of real signal near 60 Hz), but Part 2 currently reads as if the hum were in the data.
*Suggestion:* keep the steps for safety, and change the Part 2 text to say the data arrives pre-filtered.

### 7.4 · Questions for the team ❓

Bring these to Friday. Each one changes a number we will report.

1. **What claim are we testing?** Same tasks, new moments (current split)? New movements (leave-stage-out)? New
   people (needs more users)? The split should follow the claim, not the other way round.
2. **Should EMG size be comparable across stages?** If "this stage uses more muscle" is part of any result, we
   can't scale per recording. If only *patterns* matter (NMF synergy shapes), per-recording scaling is defensible.
   Which results depend on which choice?
3. **Scale all channels together, or each channel on its own?** We scale all 16 together, so a channel with
   strong contact counts for more in NMF. Classic synergy studies scale each muscle separately. Does the number of
   synergies in notebook 02 change if we do?
4. **Keep the idle-hand recordings?** They are mostly rest. They could count as useful "rest" data or as noise
   that inflates the variance explained.
5. **Is 10 Hz the right smoothing?** Many synergy studies smooth harder (around 4–6 Hz). Rerun notebook 02 with 5 Hz
   and 10 Hz: if the number of synergies changes, the result depends on a preprocessing choice and we should say so.
6. **Does EMG channel *k* sit over the same muscle on both wrists?** If one armband is mirrored relative to the
   other, the "both hands" plots in Part 6 average mismatched channels. Check the emg2pose paper/docs, or compare
   left and right correlation matrices.
7. **How should we handle the EMG→pose delay?** A fixed lag, a learned lag, or a window of past EMG?
8. **One split or cross-validation?** With test blocks as short as 8 s, how much do our scores move between folds?
   If the spread is as large as the differences we want to report, a single split is not enough.

✍️ **Q5.** Pick the one issue in 7.3 you think matters most for *our* research question, and say why in two or three
sentences.

> *Your answer:*